# Silver Layer — Products Data Cleaning & Quality Checks
**GlobalMart | Tredence DE Advanced Training**

| | |
|---|---|
| **Source** | `<your-catalog>.bronze.products` (ADLS Gen2 via Autoloader) |
| **Target** | `<your-catalog>.silver.products` |
| **SCD Type** | SCD2 — price changes over time, Gold needs "price as of order date" |

> This table is **semi-structured** — alongside flat columns (`product_id`,
> `category`, `actual_price_inr`...) it carries two **struct** columns
> (`specs`, `supplier_info`) and one **array** column (`tags`).

> **SCD2 note:** this is the *full load* notebook. Every product gets its
> first version here (`is_current = True`). The history-preserving
> `MERGE INTO` — closing out an old price version and inserting a new one
> when price changes — is built separately in the **Day 12 incremental
> notebook** (CDF-driven), not here. Same split as `silver.customers`.

### What this notebook does
| Step | Action |
|---|---|
| 1 | Setup |
| 2 | Read Bronze products + inspect schema |
| 3 | Check `_rescued_data` — confirm no silent schema drift |
| 4 | Inspect nested `specs` / `supplier_info` / `tags` |
| 5 | Investigate `specs.material` nulls — business pattern, not a DQ issue |
| 6 | Flatten structs; decide `tags`/`color_options` stay as arrays |
| 7 | DQ scan — nulls, price logic, rating range, `product_id` uniqueness |
| 8 | Transform — derive `discount_pct`, add SCD2 columns |
| 9 | Write to `silver.products` |
| 10 | Verify |

## Step 1 — Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

CATALOG      = "harsh_kumar01_npmentorskool_onmicrosoft_com"
BRONZE_TABLE = "harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.products"
SILVER_TABLE = "harsh_kumar01_npmentorskool_onmicrosoft_com.silver.products"

print(f"Reading from : {BRONZE_TABLE}")
print(f"Writing to   : {SILVER_TABLE}")

Reading from : harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.products
Writing to   : harsh_kumar01_npmentorskool_onmicrosoft_com.silver.products


## Step 2 — Read Raw Data from Bronze
Look at the actual schema before assuming anything about its shape — this
table has nested struct and array columns, so `printSchema()` matters more
here than it did for customers/orders.

In [0]:
bronze_df = spark.table(BRONZE_TABLE)
print(f"Total records in Bronze: {bronze_df.count():,}")
bronze_df.printSchema()

Total records in Bronze: 500
root
 |-- actual_price_inr: double (nullable = true)
 |-- category: string (nullable = true)
 |-- discounted_price_inr: double (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- num_ratings: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- specs: struct (nullable = true)
 |    |-- color_options: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- country_of_origin: string (nullable = true)
 |    |-- is_returnable: boolean (nullable = true)
 |    |-- material: string (nullable = true)
 |    |-- power_source: string (nullable = true)
 |    |-- return_window_days: long (nullable = true)
 |    |-- warranty_months: long (nullable = true)
 |    |-- weight_kg: double (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- supplier_info: struct (nullable = true)
 |    |-- city: string (nullable = true)


In [0]:
bronze_df.display()

actual_price_inr,category,discounted_price_inr,last_updated,num_ratings,product_id,product_name,rating,specs,sub_category,supplier_info,tags,_rescued_data,_source_file,_ingested_at
1359.0,Electronics,1216.0,2026-06-01T17:48:00Z,3097,PRD-00001,MRF Mobile Accessories - Pro Series,4.6,"List(List(Blue, Black), China, true, null, USB-C, 10, 24, 3.63)",Mobile Accessories,"List(Delhi, Greenwood Supplies, SUP-02)","List(electronics, gadget, tech, mobile-accessories)",null,/mnt/virinchy_gbmart_data/products/products_010626.json,2026-07-10T07:29:26.772Z
965.0,Electronics,887.0,2026-06-01T14:45:00Z,3449,PRD-00002,MRF Mobile Accessories - Everyday Essential,3.9,"List(List(Gray), China, true, null, USB-C, 7, 24, 4.83)",Mobile Accessories,"List(Lucknow, Wilson Solutions, SUP-10)","List(electronics, gadget, tech, mobile-accessories)",null,/mnt/virinchy_gbmart_data/products/products_010626.json,2026-07-10T07:29:26.772Z
8821.0,Electronics,8325.0,2026-06-01T16:28:00Z,2438,PRD-00003,MDH Mobile Accessories - Family Pack,4.9,"List(List(Gray), China, false, null, Battery, 10, 6, 3.35)",Mobile Accessories,"List(Kolkata, Smith & Partners, SUP-06)","List(electronics, gadget, tech, mobile-accessories)",null,/mnt/virinchy_gbmart_data/products/products_010626.json,2026-07-10T07:29:26.772Z
5054.0,Electronics,3873.0,2026-06-01T18:49:00Z,2909,PRD-00004,Lakme Mobile Accessories - Premium Edition,3.0,"List(List(Blue, Black), China, false, null, Battery, 7, 12, 2.78)",Mobile Accessories,"List(Chennai, Miller Industries, SUP-05)","List(electronics, gadget, tech, mobile-accessories)",null,/mnt/virinchy_gbmart_data/products/products_010626.json,2026-07-10T07:29:26.772Z
3057.0,Electronics,2776.0,2026-06-01T04:56:00Z,980,PRD-00005,Camlin Mobile Accessories - Heavy Duty,3.6,"List(List(Black), China, true, null, Battery, 7, 24, 3.54)",Mobile Accessories,"List(Jaipur, Davis Holdings, SUP-09)","List(electronics, gadget, tech, mobile-accessories)",null,/mnt/virinchy_gbmart_data/products/products_010626.json,2026-07-10T07:29:26.772Z
15792.0,Electronics,11282.0,2026-06-01T13:51:00Z,1272,PRD-00006,Wildcraft Mobile Accessories - Value Pack,3.4,"List(List(Black), China, true, null, Battery, 10, 24, 2.53)",Mobile Accessories,"List(Delhi, Greenwood Supplies, SUP-02)","List(electronics, gadget, tech, mobile-accessories)",null,/mnt/virinchy_gbmart_data/products/products_010626.json,2026-07-10T07:29:26.772Z
4389.0,Electronics,3273.0,2026-06-01T05:13:00Z,2440,PRD-00007,Prestige Mobile Accessories - Classic Fit,3.9,"List(List(Black), China, true, null, Battery, 30, 24, 4.63)",Mobile Accessories,"List(Kolkata, Smith & Partners, SUP-06)","List(electronics, gadget, tech, mobile-accessories)",null,/mnt/virinchy_gbmart_data/products/products_010626.json,2026-07-10T07:29:26.772Z
10752.0,Electronics,7606.0,2026-06-01T18:57:00Z,2617,PRD-00008,Decathlon Mobile Accessories - Multi-Color Pack,3.6,"List(List(Gray), China, true, null, AC Adapter, 15, 6, 2.28)",Mobile Accessories,"List(Jaipur, Davis Holdings, SUP-09)","List(electronics, gadget, tech, mobile-accessories)",null,/mnt/virinchy_gbmart_data/products/products_010626.json,2026-07-10T07:29:26.772Z
6734.0,Electronics,5508.0,2026-06-01T11:06:00Z,2561,PRD-00009,Bajaj Mobile Accessories - Pro Series,3.2,"List(List(Gray), China, false, null, AC Adapter, 15, 6, 3.95)",Mobile Accessories,"List(Kolkata, Smith & Partners, SUP-06)","List(electronics, gadget, tech, mobile-accessories)",null,/mnt/virinchy_gbmart_data/products/products_010626.json,2026-07-10T07:29:26.772Z
8725.0,Electronics,5908.0,2026-06-01T13:42:00Z,3948,PRD-00010,Havells Mobile Accessories - New Arrival,3.5,"List(List(Black, Silver), China, false, null, USB-C, 30, 12, 0.53)",Mobile Accessories,"List(Mumbai, Johnson & Co., SUP-01)","List(electronics, gadget, tech, mobile-accessories)",null,/mnt/virinchy_gbmart_data/products/products_010626.json,2026-07-10T07:29:26.772Z


## Step 3 — Investigate: `_rescued_data`

Autoloader writes any field it **couldn't match to the inferred schema**
into `_rescued_data` instead of silently dropping it. We check the full
table, and keep this check permanently in the pipeline — a future file
with an extra/renamed field would land here silently if we ever stopped
looking.

In [0]:
rescued_count = bronze_df.filter(col("_rescued_data").isNotNull()).count()
print(f"Rows with non-null _rescued_data: {rescued_count:,} / {bronze_df.count():,}")

if rescued_count > 0:
    bronze_df.filter(col("_rescued_data").isNotNull()).select("product_id", "_rescued_data", "_source_file").display()
else:
    print("No schema drift detected — every row matched the inferred schema.")

Rows with non-null _rescued_data: 0 / 500
No schema drift detected — every row matched the inferred schema.


**Finding:** 0 of 500 rows have non-null `_rescued_data`. No schema drift.

## Step 4 — Investigate: What's Actually Inside `specs` and `supplier_info`?
Before deciding how to flatten anything, look at the full set of
sub-fields Spark parsed, and whether `tags` has constant or variable
cardinality (that decides array vs. bridge table).

In [0]:
print("=== specs struct fields ===")
bronze_df.select("specs.*").printSchema()

print("=== supplier_info struct fields ===")
bronze_df.select("supplier_info.*").printSchema()

bronze_df.select("product_id", "specs.*").show(5, truncate=False)
bronze_df.select("product_id", "supplier_info.*").show(5, truncate=False)

bronze_df.select(size("tags").alias("tag_count")).groupBy("tag_count").count().orderBy("tag_count").display()

=== specs struct fields ===
root
 |-- color_options: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- country_of_origin: string (nullable = true)
 |-- is_returnable: boolean (nullable = true)
 |-- material: string (nullable = true)
 |-- power_source: string (nullable = true)
 |-- return_window_days: long (nullable = true)
 |-- warranty_months: long (nullable = true)
 |-- weight_kg: double (nullable = true)

=== supplier_info struct fields ===
root
 |-- city: string (nullable = true)
 |-- name: string (nullable = true)
 |-- supplier_id: string (nullable = true)

+----------+-------------+-----------------+-------------+--------+------------+------------------+---------------+---------+
|product_id|color_options|country_of_origin|is_returnable|material|power_source|return_window_days|warranty_months|weight_kg|
+----------+-------------+-----------------+-------------+--------+------------+------------------+---------------+---------+
|PRD-00001 |[Blue, Black]|

tag_count,count
4,500


**Findings:**
- `is_returnable` is already `boolean`, `return_window_days` is already
  `long` — no type casts needed (unlike PhoneNumber/₹ in earlier notebooks).
- `tag_count` is a constant **4** across all 500 rows — no variable
  cardinality, so `tags` stays a simple `array<string>`, no bridge table.

## Step 5 — Investigate: Is `specs.material` Always Null?
The sample rows all show `material = NULL`. Check across the full table
and see if it correlates with `sub_category` before deciding anything.

In [0]:
total = bronze_df.count()
material_null = bronze_df.filter(col("specs.material").isNull()).count()
print(f"NULL material: {material_null:,} / {total:,}")

bronze_df.groupBy("sub_category") \
    .agg(
        count(when(col("specs.material").isNull(), 1)).alias("null_material"),
        count(when(col("specs.material").isNotNull(), 1)).alias("has_material"),
        count("*").alias("total")
    ).display()

NULL material: 400 / 500


sub_category,null_material,has_material,total
Breakfast Foods,10,0,10
Cooking Essentials,10,0,10
Organic Products,10,0,10
Tools & Equipment,10,0,10
Smart Home Devices,10,0,10
Car Accessories,10,0,10
Personal Care Appliances,10,0,10
Cleaning Appliances,10,0,10
Skincare,10,0,10
Climate Control,10,0,10


### Finding: `material` is Null by Category Design, Not by Error

`material` is populated **only** for 10 sub-categories — all of them
apparel/home-goods where "material" is a meaningful attribute (Clothing,
Footwear, Bags & Luggage, Furniture, Home Decor, Bedding & Linen, Storage,
Lighting, Watches & Accessories). Electronics, personal care, food, and
tools genuinely don't have a "material" attribute the way clothing or
furniture do. The split is perfectly clean — 10/10 either way per
sub-category, no partial nulls within a sub-category.

**Decision: Keep `material` nullable. Not a DQ issue — no quarantine.**

## Step 6 — Flatten Nested Columns

### specs and supplier_info → top-level columns
Both are clean structs — no further DQ surprises. We flatten with
`.specs.*` / `.supplier_info.*` selects.

### tags and color_options → stay as array<string>
Neither is a many-to-many relationship between two separate entities (that
would need a bridge table, like `bridge_product_promotion`). They're
multi-valued **attributes of one product** — a product simply has several
colors or several tags. Delta supports `array<string>` natively, so we
keep them as-is rather than exploding (which would break the 1-row-per-
product grain) or forcing them into a relational shape they don't need.

In [0]:
flattened_df = bronze_df.select(
    "product_id", "product_name", "category", "sub_category",
    "actual_price_inr", "discounted_price_inr", "rating", "num_ratings",
    "last_updated",

    col("specs.power_source").alias("power_source"),
    col("specs.country_of_origin").alias("country_of_origin"),
    col("specs.color_options").alias("color_options"),
    col("specs.is_returnable").alias("is_returnable"),
    col("specs.return_window_days").alias("return_window_days"),
    col("specs.material").alias("material"),
    col("specs.warranty_months").alias("warranty_months"),
    col("specs.weight_kg").alias("weight_kg"),

    col("supplier_info.supplier_id").alias("supplier_id"),
    col("supplier_info.name").alias("supplier_name"),
    col("supplier_info.city").alias("supplier_city"),

    "tags",
    "_source_file"
)

flattened_df.printSchema()
flattened_df.display()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- actual_price_inr: double (nullable = true)
 |-- discounted_price_inr: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- num_ratings: long (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- power_source: string (nullable = true)
 |-- country_of_origin: string (nullable = true)
 |-- color_options: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- is_returnable: boolean (nullable = true)
 |-- return_window_days: long (nullable = true)
 |-- material: string (nullable = true)
 |-- warranty_months: long (nullable = true)
 |-- weight_kg: double (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- supplier_city: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (co

product_id,product_name,category,sub_category,actual_price_inr,discounted_price_inr,rating,num_ratings,last_updated,power_source,country_of_origin,color_options,is_returnable,return_window_days,material,warranty_months,weight_kg,supplier_id,supplier_name,supplier_city,tags,_source_file
PRD-00001,MRF Mobile Accessories - Pro Series,Electronics,Mobile Accessories,1359.0,1216.0,4.6,3097,2026-06-01T17:48:00Z,USB-C,China,"List(Blue, Black)",true,10,null,24,3.63,SUP-02,Greenwood Supplies,Delhi,"List(electronics, gadget, tech, mobile-accessories)",/mnt/virinchy_gbmart_data/products/products_010626.json
PRD-00002,MRF Mobile Accessories - Everyday Essential,Electronics,Mobile Accessories,965.0,887.0,3.9,3449,2026-06-01T14:45:00Z,USB-C,China,List(Gray),true,7,null,24,4.83,SUP-10,Wilson Solutions,Lucknow,"List(electronics, gadget, tech, mobile-accessories)",/mnt/virinchy_gbmart_data/products/products_010626.json
PRD-00003,MDH Mobile Accessories - Family Pack,Electronics,Mobile Accessories,8821.0,8325.0,4.9,2438,2026-06-01T16:28:00Z,Battery,China,List(Gray),false,10,null,6,3.35,SUP-06,Smith & Partners,Kolkata,"List(electronics, gadget, tech, mobile-accessories)",/mnt/virinchy_gbmart_data/products/products_010626.json
PRD-00004,Lakme Mobile Accessories - Premium Edition,Electronics,Mobile Accessories,5054.0,3873.0,3.0,2909,2026-06-01T18:49:00Z,Battery,China,"List(Blue, Black)",false,7,null,12,2.78,SUP-05,Miller Industries,Chennai,"List(electronics, gadget, tech, mobile-accessories)",/mnt/virinchy_gbmart_data/products/products_010626.json
PRD-00005,Camlin Mobile Accessories - Heavy Duty,Electronics,Mobile Accessories,3057.0,2776.0,3.6,980,2026-06-01T04:56:00Z,Battery,China,List(Black),true,7,null,24,3.54,SUP-09,Davis Holdings,Jaipur,"List(electronics, gadget, tech, mobile-accessories)",/mnt/virinchy_gbmart_data/products/products_010626.json
PRD-00006,Wildcraft Mobile Accessories - Value Pack,Electronics,Mobile Accessories,15792.0,11282.0,3.4,1272,2026-06-01T13:51:00Z,Battery,China,List(Black),true,10,null,24,2.53,SUP-02,Greenwood Supplies,Delhi,"List(electronics, gadget, tech, mobile-accessories)",/mnt/virinchy_gbmart_data/products/products_010626.json
PRD-00007,Prestige Mobile Accessories - Classic Fit,Electronics,Mobile Accessories,4389.0,3273.0,3.9,2440,2026-06-01T05:13:00Z,Battery,China,List(Black),true,30,null,24,4.63,SUP-06,Smith & Partners,Kolkata,"List(electronics, gadget, tech, mobile-accessories)",/mnt/virinchy_gbmart_data/products/products_010626.json
PRD-00008,Decathlon Mobile Accessories - Multi-Color Pack,Electronics,Mobile Accessories,10752.0,7606.0,3.6,2617,2026-06-01T18:57:00Z,AC Adapter,China,List(Gray),true,15,null,6,2.28,SUP-09,Davis Holdings,Jaipur,"List(electronics, gadget, tech, mobile-accessories)",/mnt/virinchy_gbmart_data/products/products_010626.json
PRD-00009,Bajaj Mobile Accessories - Pro Series,Electronics,Mobile Accessories,6734.0,5508.0,3.2,2561,2026-06-01T11:06:00Z,AC Adapter,China,List(Gray),false,15,null,6,3.95,SUP-06,Smith & Partners,Kolkata,"List(electronics, gadget, tech, mobile-accessories)",/mnt/virinchy_gbmart_data/products/products_010626.json
PRD-00010,Havells Mobile Accessories - New Arrival,Electronics,Mobile Accessories,8725.0,5908.0,3.5,3948,2026-06-01T13:42:00Z,USB-C,China,"List(Black, Silver)",false,30,null,12,0.53,SUP-01,Johnson & Co.,Mumbai,"List(electronics, gadget, tech, mobile-accessories)",/mnt/virinchy_gbmart_data/products/products_010626.json


### Querying `tags` (array<string>) — Business Examples
Showing participants how to query an array column without exploding the
table itself — `explode()` only reshapes the query result, not storage.


In [0]:
# Business Q: "Show me all products tagged as electronics"
flattened_df.filter(array_contains(col("tags"), "electronics")) \
    .select("product_id", "product_name", "tags").display()


product_id,product_name,tags
PRD-00001,MRF Mobile Accessories - Pro Series,"List(electronics, gadget, tech, mobile-accessories)"
PRD-00002,MRF Mobile Accessories - Everyday Essential,"List(electronics, gadget, tech, mobile-accessories)"
PRD-00003,MDH Mobile Accessories - Family Pack,"List(electronics, gadget, tech, mobile-accessories)"
PRD-00004,Lakme Mobile Accessories - Premium Edition,"List(electronics, gadget, tech, mobile-accessories)"
PRD-00005,Camlin Mobile Accessories - Heavy Duty,"List(electronics, gadget, tech, mobile-accessories)"
PRD-00006,Wildcraft Mobile Accessories - Value Pack,"List(electronics, gadget, tech, mobile-accessories)"
PRD-00007,Prestige Mobile Accessories - Classic Fit,"List(electronics, gadget, tech, mobile-accessories)"
PRD-00008,Decathlon Mobile Accessories - Multi-Color Pack,"List(electronics, gadget, tech, mobile-accessories)"
PRD-00009,Bajaj Mobile Accessories - Pro Series,"List(electronics, gadget, tech, mobile-accessories)"
PRD-00010,Havells Mobile Accessories - New Arrival,"List(electronics, gadget, tech, mobile-accessories)"


In [0]:
# Business Q: "How many tags does each product have?"
flattened_df.select("product_id", "tags", size("tags").alias("tag_count")).display()


product_id,tags,tag_count
PRD-00001,"List(electronics, gadget, tech, mobile-accessories)",4
PRD-00002,"List(electronics, gadget, tech, mobile-accessories)",4
PRD-00003,"List(electronics, gadget, tech, mobile-accessories)",4
PRD-00004,"List(electronics, gadget, tech, mobile-accessories)",4
PRD-00005,"List(electronics, gadget, tech, mobile-accessories)",4
PRD-00006,"List(electronics, gadget, tech, mobile-accessories)",4
PRD-00007,"List(electronics, gadget, tech, mobile-accessories)",4
PRD-00008,"List(electronics, gadget, tech, mobile-accessories)",4
PRD-00009,"List(electronics, gadget, tech, mobile-accessories)",4
PRD-00010,"List(electronics, gadget, tech, mobile-accessories)",4


In [0]:
# Business Q: "Break tags into one row each, so I can count products per tag"
flattened_df.select("product_id", "product_name", explode("tags").alias("tag")).display()


product_id,product_name,tag
PRD-00001,MRF Mobile Accessories - Pro Series,electronics
PRD-00001,MRF Mobile Accessories - Pro Series,gadget
PRD-00001,MRF Mobile Accessories - Pro Series,tech
PRD-00001,MRF Mobile Accessories - Pro Series,mobile-accessories
PRD-00002,MRF Mobile Accessories - Everyday Essential,electronics
PRD-00002,MRF Mobile Accessories - Everyday Essential,gadget
PRD-00002,MRF Mobile Accessories - Everyday Essential,tech
PRD-00002,MRF Mobile Accessories - Everyday Essential,mobile-accessories
PRD-00003,MDH Mobile Accessories - Family Pack,electronics
PRD-00003,MDH Mobile Accessories - Family Pack,gadget


## Step 7 — First DQ Scan

| Rule | What we check | Why |
|---|---|---|
| `NULL_PRODUCT_ID` | `product_id` is null | Primary key |
| `NULL_PRODUCT_NAME` | `product_name` is null | Required for catalog/display |
| `NULL_CATEGORY` | `category` is null | Drives `dim_product` hierarchy |
| `INVALID_PRICE` | `actual_price_inr` is null or <= 0 | Can't have zero/negative price |
| `DISCOUNT_EXCEEDS_PRICE` | `discounted_price_inr > actual_price_inr` | Can't discount to more than the original price |
| `INVALID_RATING` | `rating` not in `[0, 5]` | Standard rating scale |
| `NEGATIVE_NUM_RATINGS` | `num_ratings < 0` | Count can't be negative |
| `NULL_SUPPLIER_ID` | `supplier_id` is null | Needed for FK in Gold |

In [0]:
dq_scan_df = flattened_df \
    .withColumn("_dq_issue",
        when(col("product_id").isNull(),                                          lit("NULL_PRODUCT_ID"))
        .when(col("product_name").isNull(),                                       lit("NULL_PRODUCT_NAME"))
        .when(col("category").isNull(),                                           lit("NULL_CATEGORY"))
        .when(col("actual_price_inr").isNull() | (col("actual_price_inr") <= 0),   lit("INVALID_PRICE"))
        .when(col("discounted_price_inr") > col("actual_price_inr"),               lit("DISCOUNT_EXCEEDS_PRICE"))
        .when(~col("rating").between(0, 5),                                        lit("INVALID_RATING"))
        .when(col("num_ratings") < 0,                                              lit("NEGATIVE_NUM_RATINGS"))
        .when(col("supplier_id").isNull(),                                        lit("NULL_SUPPLIER_ID"))
        .otherwise(lit(None))
    )

print("=== DQ Issues Found ===")
dq_scan_df.groupBy("_dq_issue").count().orderBy("count", ascending=False).display()

=== DQ Issues Found ===


_dq_issue,count
null,500


In [0]:
# product_id uniqueness
dupe_count = flattened_df.groupBy("product_id").count().filter("count > 1").count()
print(f"Duplicate product_id count: {dupe_count}")

# discounted_price_inr nulls
null_discount_count = flattened_df.filter(col("discounted_price_inr").isNull()).count()
print(f"NULL discounted_price_inr count: {null_discount_count}")

Duplicate product_id count: 0
NULL discounted_price_inr count: 0


In [0]:
# Business Q: "Which tags are most commonly used across the catalog?"
flattened_df.select("product_id", explode("tags").alias("tag")) \
    .groupBy("tag") \
    .count() \
    .orderBy(col("count").desc()) \
    .display()


tag,count
home,100
furniture,60
gadget,50
automotive,50
education,50
beauty,50
food,50
gourmet,50
stationery,50
kitchen,50


**Finding: 0 DQ issues, 0 duplicate `product_id`, 0 null
`discounted_price_inr`.** Products is the first dataset in this program
that needs no quarantine table — `flattened_df` is already our clean
dataset.

## Step 8 — Transform

| Column | Transformation | Business Need |
|---|---|---|
| `discount_pct` | `(actual_price_inr - discounted_price_inr) / actual_price_inr * 100` | Pre-computed for Gold reporting |
| `product_sk` | `sha2(product_id + effective_start_date)` | Surrogate key for SCD2 |
| `effective_start_date` | `current_date()` | Version start |
| `effective_end_date` | `NULL` | Open-ended while current |
| `is_current` | `True` | Active version on initial load |

In [0]:
silver_df = (
    flattened_df
    .withColumn("discount_pct",
        round((col("actual_price_inr") - col("discounted_price_inr")) / col("actual_price_inr") * 100, 2)
    )
    .withColumn("effective_start_date", current_date())
    .withColumn("effective_end_date", lit(None).cast(DateType()))
    .withColumn("is_current", lit(True))
    .withColumn("product_sk",
        sha2(concat_ws("|", col("product_id"), col("effective_start_date").cast("string")), 256)
    )
    .withColumn("_silver_updated_at", current_timestamp())
    .select(
        "product_sk", "product_id", "product_name", "category", "sub_category",
        "actual_price_inr", "discounted_price_inr", "discount_pct",
        "rating", "num_ratings",
        "power_source", "country_of_origin", "color_options", "is_returnable",
        "return_window_days", "material", "warranty_months", "weight_kg",
        "supplier_id", "supplier_name", "supplier_city",
        "tags",
        "is_current", "effective_start_date", "effective_end_date",
        "last_updated", "_silver_updated_at"
    )
)

silver_df.display()

product_sk,product_id,product_name,category,sub_category,actual_price_inr,discounted_price_inr,discount_pct,rating,num_ratings,power_source,country_of_origin,color_options,is_returnable,return_window_days,material,warranty_months,weight_kg,supplier_id,supplier_name,supplier_city,tags,is_current,effective_start_date,effective_end_date,last_updated,_silver_updated_at
360a574752eb92469dbd127c002211ce2202f134a9db1f1f74c8e0c4f5602183,PRD-00001,MRF Mobile Accessories - Pro Series,Electronics,Mobile Accessories,1359.0,1216.0,10.52,4.6,3097,USB-C,China,"List(Blue, Black)",true,10,null,24,3.63,SUP-02,Greenwood Supplies,Delhi,"List(electronics, gadget, tech, mobile-accessories)",true,2026-07-10,null,2026-06-01T17:48:00Z,2026-07-10T12:53:22.93753Z
1a3746bc0a257ea7adff993c42f173fa8f71d65375c96ab423c1a364152733f6,PRD-00002,MRF Mobile Accessories - Everyday Essential,Electronics,Mobile Accessories,965.0,887.0,8.08,3.9,3449,USB-C,China,List(Gray),true,7,null,24,4.83,SUP-10,Wilson Solutions,Lucknow,"List(electronics, gadget, tech, mobile-accessories)",true,2026-07-10,null,2026-06-01T14:45:00Z,2026-07-10T12:53:22.93753Z
a9a7615b1a6ff554b472a610b807878337de63eeefa87e287d0a84ca99d8b74f,PRD-00003,MDH Mobile Accessories - Family Pack,Electronics,Mobile Accessories,8821.0,8325.0,5.62,4.9,2438,Battery,China,List(Gray),false,10,null,6,3.35,SUP-06,Smith & Partners,Kolkata,"List(electronics, gadget, tech, mobile-accessories)",true,2026-07-10,null,2026-06-01T16:28:00Z,2026-07-10T12:53:22.93753Z
e98ba313221cc602b9bacf42f1b9f8084c5f825805b94999fbfa9bbb97a90391,PRD-00004,Lakme Mobile Accessories - Premium Edition,Electronics,Mobile Accessories,5054.0,3873.0,23.37,3.0,2909,Battery,China,"List(Blue, Black)",false,7,null,12,2.78,SUP-05,Miller Industries,Chennai,"List(electronics, gadget, tech, mobile-accessories)",true,2026-07-10,null,2026-06-01T18:49:00Z,2026-07-10T12:53:22.93753Z
412061ac27e6148a913dddd6b0144d823c6096c2fa2fca69df4a18de72deeb46,PRD-00005,Camlin Mobile Accessories - Heavy Duty,Electronics,Mobile Accessories,3057.0,2776.0,9.19,3.6,980,Battery,China,List(Black),true,7,null,24,3.54,SUP-09,Davis Holdings,Jaipur,"List(electronics, gadget, tech, mobile-accessories)",true,2026-07-10,null,2026-06-01T04:56:00Z,2026-07-10T12:53:22.93753Z
6ac48fe8c42dde2dbcabb2f61abad363e71a0730b944cde69ddffaa984431011,PRD-00006,Wildcraft Mobile Accessories - Value Pack,Electronics,Mobile Accessories,15792.0,11282.0,28.56,3.4,1272,Battery,China,List(Black),true,10,null,24,2.53,SUP-02,Greenwood Supplies,Delhi,"List(electronics, gadget, tech, mobile-accessories)",true,2026-07-10,null,2026-06-01T13:51:00Z,2026-07-10T12:53:22.93753Z
ebf394a40ac32edec91cbbc011a600a6e88936fed7a8012a1bc1d6bde2713a91,PRD-00007,Prestige Mobile Accessories - Classic Fit,Electronics,Mobile Accessories,4389.0,3273.0,25.43,3.9,2440,Battery,China,List(Black),true,30,null,24,4.63,SUP-06,Smith & Partners,Kolkata,"List(electronics, gadget, tech, mobile-accessories)",true,2026-07-10,null,2026-06-01T05:13:00Z,2026-07-10T12:53:22.93753Z
1530db070d75b26e618183118f06db192d0673bd817589e35a44dac50f06977f,PRD-00008,Decathlon Mobile Accessories - Multi-Color Pack,Electronics,Mobile Accessories,10752.0,7606.0,29.26,3.6,2617,AC Adapter,China,List(Gray),true,15,null,6,2.28,SUP-09,Davis Holdings,Jaipur,"List(electronics, gadget, tech, mobile-accessories)",true,2026-07-10,null,2026-06-01T18:57:00Z,2026-07-10T12:53:22.93753Z
bda2cfeb9c4304cce5f4b6268e1f8bb1643d94e22f71a6e6ac2347b08595c67e,PRD-00009,Bajaj Mobile Accessories - Pro Series,Electronics,Mobile Accessories,6734.0,5508.0,18.21,3.2,2561,AC Adapter,China,List(Gray),false,15,null,6,3.95,SUP-06,Smith & Partners,Kolkata,"List(electronics, gadget, tech, mobile-accessories)",true,2026-07-10,null,2026-06-01T11:06:00Z,2026-07-10T12:53:22.93753Z
435d324e938bfff2d89aff25a8cbf30da5c49f96ed336ecf3cfac6700af5aefb,PRD-00010,Havells Mobile Accessories - New Arrival,Electronics,Mobile Accessories,8725.0,5908.0,32.29,3.5,3948,USB-C,China,"List(Black, Silver)",false,30,null,12,

## Step 9 — Write to Silver

Initial full load — every product gets its first SCD2 version
(`is_current = True`, `effective_end_date = NULL`). No quarantine table is
needed since no rows failed DQ checks.

The incremental version of this pipeline (Day 12) will read Bronze CDF
changes and `MERGE` them in — closing out old versions when price changes
and inserting new ones. That logic is intentionally not built here.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f"Written to {SILVER_TABLE}: {spark.table(SILVER_TABLE).count():,} rows")

Written to harsh_kumar01_npmentorskool_onmicrosoft_com.silver.products: 500 rows


## Step 10 — Verify

In [0]:
df = spark.table(SILVER_TABLE)
print(f"silver.products rows : {df.count():,}")
df.printSchema()

df.groupBy("category").count().orderBy("count", ascending=False).display()
df.select(avg("discount_pct").alias("avg_discount_pct"), avg("rating").alias("avg_rating")).display()

silver.products rows : 500
root
 |-- product_sk: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- actual_price_inr: double (nullable = true)
 |-- discounted_price_inr: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- num_ratings: long (nullable = true)
 |-- power_source: string (nullable = true)
 |-- country_of_origin: string (nullable = true)
 |-- color_options: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- is_returnable: boolean (nullable = true)
 |-- return_window_days: long (nullable = true)
 |-- material: string (nullable = true)
 |-- warranty_months: long (nullable = true)
 |-- weight_kg: double (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- supplier_city: string (nullable = t

category,count
Beauty & Personal Care,50
Fashion,50
Sports & Fitness,50
Books & Stationery,50
Electronics,50
Automotive,50
Grocery & Gourmet,50
Appliances,50
Home & Furniture,50
Toys & Baby,50


avg_discount_pct,avg_rating
20.04082,4.004999999999998


## Reset (if needed)

In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")
# print("Reset complete")